# Recuperación semántica mediante SigLIP

Compara líneas base, una vista y agregación multivista. El dataset de habitaciones cubre idioma, paráfrasis y rechazo; Matterport3D/R2R aporta viewpoints físicos, topología y separación por edificio.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_navigation_ws" / "src").is_dir())
sys.path.insert(0, str(repo / "experiments" / "shared"))

import os
import time
import numpy as np
import pandas as pd

from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from plotting import METHOD_LABELS
from semantic_evaluation.core.dataset_adapters import load_dataset
from semantic_evaluation.core.offline_dataset import load_queries

ctx = bootstrap_offline()
config = ctx["config"]
bundles = {spec.dataset_id: load_dataset(spec, ctx["repo_root"])
           for spec in ctx["dataset_specs"]}
specs = {spec.dataset_id: spec for spec in ctx["dataset_specs"]}
display(pd.DataFrame([{"dataset_id": key, "available": not bundle.skipped,
                       "nodes": len(bundle.nodes), "reason": bundle.skip_reason}
                      for key, bundle in bundles.items()]))


## Codificación y caché


In [ ]:
from semantic_evaluation.core import EmbeddingCache
from semantic_evaluation.core.offline_encoding import encode_observations, embed_texts
from semantic_vision_core import SemanticVisionPipeline

siglip = config["models"]["siglip"]
cache_root = resolve_repo_path(ctx["repo_root"], config["paths"]["cache_root"])
cache = EmbeddingCache(str(cache_root / "siglip"))
pipeline = None
model_error = None
try:
    pipeline = SemanticVisionPipeline(
        retrieval_mode="siglip_pure", siglip_model_id=siglip["model_id"],
        device=ctx["device"], processor_fast=bool(siglip["processor_fast"]),
        local_files_only=bool(siglip.get("local_files_only", False)))
except (ImportError, OSError, RuntimeError) as error:
    model_error = str(error)
    print("SigLIP no disponible; inferencia omitida:", model_error)


## Métodos y métricas


In [ ]:
from semantic_evaluation.core.experiment_runner import prepare_queries, run_method
from semantic_evaluation.core.retrieval_metrics import results_to_rows, summarize, topological_distance
from semantic_navigation_core.multiview import MultiviewConfig, SUPPORTED_AGGREGATIONS
from semantic_navigation_core.retrieval import (
    METHOD_RANDOM_BASELINE, METHOD_ROOM_LABEL_BASELINE,
    METHOD_SINGLE_VIEW_SIGLIP, METHOD_MULTIVIEW_SIGLIP, RetrievalConfig,
)

mv = config["retrieval"]["multiview"]
initial_threshold = config["retrieval"]["rejection"]["initial_threshold"]
all_results = []
if pipeline is not None:
    for dataset_id in ("siglip_rooms", "sunrgbd", "matterport3d"):
        bundle = bundles[dataset_id]
        if bundle.skipped or not bundle.nodes:
            continue
        encode_observations(bundle.nodes, pipeline, cache, siglip["model_id"])
        queries = load_queries(str(resolve_repo_path(ctx["repo_root"], specs[dataset_id].queries_file)))
        prepared_all = prepare_queries(queries, bundle)
        prepared = [
            item for item in prepared_all
            if item.query.is_negative or item.valid_node_ids
        ]
        omitted = len(prepared_all) - len(prepared)
        if omitted:
            print(
                f"{dataset_id}: se omiten {omitted} consultas positivas sin "
                "valid_node_ids explícitos; expected_room no es ground truth de instancia."
            )
        if not prepared:
            continue
        embeddings = embed_texts([item.query.text for item in prepared], pipeline, cache,
                                 siglip["model_id"])
        for item, embedding in zip(prepared, embeddings):
            item.embedding = embedding
        methods = {
            "random_baseline": RetrievalConfig(method=METHOD_RANDOM_BASELINE,
                                                 seed=config["experiment"]["seed"]),
            "room_label_baseline": RetrievalConfig(method=METHOD_ROOM_LABEL_BASELINE),
            "single_view_siglip": RetrievalConfig(method=METHOD_SINGLE_VIEW_SIGLIP),
        }
        for label, method_config in methods.items():
            all_results.extend(run_method(label, prepared, bundle, method_config,
                                          initial_threshold))
        if dataset_id == "matterport3d":
            for aggregation in SUPPORTED_AGGREGATIONS:
                method_config = RetrievalConfig(
                    method=METHOD_MULTIVIEW_SIGLIP,
                    multiview=MultiviewConfig(
                        method=aggregation, top_k=mv["top_k"],
                        max_weight=mv["max_weight"], topk_weight=mv["topk_weight"]))
                all_results.extend(run_method(
                    f"multiview_siglip_{aggregation}", prepared, bundle,
                    method_config, initial_threshold))

case_rows = results_to_rows(all_results)
for row, result in zip(case_rows, all_results):
    bundle = bundles[result.dataset_id or result.scene_id]
    distances = [topological_distance(bundle.topology_edges, result.top1_id, valid)
                 for valid in result.valid_node_ids]
    row["topological_distance"] = min(
        (distance for distance in distances if distance is not None), default=np.nan)
cases = pd.DataFrame(case_rows)
summary = pd.DataFrame(summarize(all_results, ("dataset_id", "method", "query_type", "language")))
display(summary)
print("Caché en esta ejecución:", cache.stats())


## Resultados por edificio y agregación


In [ ]:
if not cases.empty:
    cases["building_id"] = cases["predicted_node_id"].map(
        lambda value: value.split(":", 1)[0] if isinstance(value, str) and ":" in value else None)
    display(cases.groupby(["dataset_id", "method"], dropna=False)[
        ["recall_at_1", "recall_at_3", "recall_at_5", "reciprocal_rank",
         "retrieval_latency_ms", "topological_distance"]].mean(numeric_only=True))
else:
    print("No hay resultados: falta un checkpoint SigLIP local o consultas anotadas.")


## Idioma, paráfrasis y rechazo negativo


In [ ]:
if not cases.empty:
    display(cases.groupby(["language", "query_type", "method"], dropna=False)[
        ["recall_at_1", "reciprocal_rank"]].mean(numeric_only=True))
    negatives = cases.loc[cases["is_negative"]]
    if not negatives.empty:
        display(negatives.groupby(["language", "method"])["rejected"].mean())
    else:
        print("No hay consultas negativas ejecutadas.")


## Exportación e interpretación


In [ ]:
from reproducibility import collect_manifest, save_manifest

results_root = resolve_repo_path(ctx["repo_root"], config["paths"]["results_root"]) / "siglip_retrieval"
results_root.mkdir(parents=True, exist_ok=True)
if not cases.empty:
    cases.to_csv(results_root / "cases.csv", index=False)
    summary.to_csv(results_root / "summary.csv", index=False)
manifest = collect_manifest(config, repo_dir=str(ctx["repo_root"]), device=ctx["device"],
                            extra={"notebook": "01_siglip_retrieval",
                                   "cache": cache.stats(), "n_cases": len(cases)})
save_manifest(str(results_root / "manifest.json"), manifest)
if summary.empty:
    print("No hay valores calculados que interpretar.")
else:
    best = summary.dropna(subset=["recall_at_1"]).sort_values("recall_at_1", ascending=False).head(1)
    if not best.empty:
        row = best.iloc[0]
        print(f"La mayor Recall@1 observada fue {row['recall_at_1']:.3f} "
              f"para {row['method']} en {row['dataset_id']} ({row['query_type']}, {row['language']}).")
